# 01 prep: collect everything (internet ON)

Part of the TargetGate pre-registered study (draft v2).
This notebook is the ONLY one allowed to use the network.
It downloads wheels and model weights, records exact versions, and writes
a manifest. Save the output as a Kaggle dataset, then run
`02_pilot` with internet OFF.

Rules for this notebook:
- Keep the notebook PRIVATE.
- Put no test attacks and no secrets in it. Hugging Face token goes in
  Kaggle Secrets (Add-ons > Secrets), never in a cell.
- Do not print the token.


In [1]:
# 1. record the environment
import json, os, platform, subprocess, sys, datetime

def sh(cmd):
    """Run a shell command and return its stripped output."""
    return subprocess.run(
        cmd, shell=True, capture_output=True, text=True
    ).stdout.strip()

ENV = {
    'utc_time': datetime.datetime.utcnow().isoformat() + 'Z',
    'python': sys.version,
    'platform': platform.platform(),
    'gpu': sh('nvidia-smi --query-gpu=name,memory.total,driver_version '
              '--format=csv,noheader'),
    'cuda': sh('nvcc --version | tail -1'),
}
print(json.dumps(ENV, indent=2))


{
  "utc_time": "2026-11-23T15:45:01.661444Z",
  "python": "3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]",
  "platform": "Linux-6.18.55+-x86_64-with-glibc2.39",
  "gpu": "Tesla T4, 15360 MiB, 580.178.04\nTesla T4, 15360 MiB, 580.178.04",
  "cuda": "Build cuda_13.0.r13.0/compiler.36424714_0"
}


/tmp/ipykernel_273/2696704461.py:11: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'utc_time': datetime.datetime.utcnow().isoformat() + 'Z',


In [2]:
# 2. choices fixed here. Edit, then they go into the manifest.
AGENTDOJO_VERSION = '0.1.35'   # pinned; current release when written

# One entry per candidate family. Fill model_id with the exact repo you
# will test. Leave None to skip. Do NOT choose by defence results.
# revision=None means: pin the latest commit at download time and
# record its SHA in MANIFEST.json.
# Full-precision 12B-14B repos are 24-30 GB: too big for the 20 GB
# output limit and for one 16 GB T4. Use a 4-bit (AWQ/GPTQ) repo.
# Entries set to None need a 4-bit repo found with the discovery cell.
CANDIDATES = {
    'deepseek': {'model_id': None, 'revision': None},  # need 4-bit repo
    'qwen':     {'model_id': 'Qwen/Qwen3-14B-AWQ', 'revision': None},
    'mistral':  {'model_id': None, 'revision': None},  # need 4-bit repo
    'gemma':    {'model_id': None, 'revision': None},  # need 4-bit repo
    'llama':    {'model_id': None, 'revision': None},  # need 4-bit repo
    'phi':      {'model_id': None, 'revision': None},  # need 4-bit repo
}
# 7B to 14B only. State the quantisation in the model_id.
# Disk budget: Kaggle output limit is 20 GB. Keep 2 GB headroom.
# Download ONE or TWO families per run, then save a dataset, then
# repeat with a fresh notebook version for the next family.
ACTIVE = ['qwen']        # change to the family you want this run
MAX_BUNDLE_GB = 18
OUT = '/kaggle/working/bundle'
os.makedirs(OUT, exist_ok=True)

# guard: warn if an active model does not look quantised
QUANT_MARKERS = ('awq', 'gptq', 'int4', '4bit', '4-bit', 'gguf', 'q4')
for _family in ACTIVE:
    _id = (CANDIDATES[_family]['model_id'] or '').lower()
    if not any(marker in _id for marker in QUANT_MARKERS):
        print(f'WARNING: {_family} ({_id or "empty"}) does not look '
              'quantised; it may not fit disk or GPU')

In [3]:
# 3. wheels for offline install (agentdojo and its dependencies)
WHEELS = os.path.join(OUT, 'wheels')
os.makedirs(WHEELS, exist_ok=True)
print(sh(
    f'pip download agentdojo=={AGENTDOJO_VERSION} huggingface_hub '
    f'-d {WHEELS} -q 2>&1 | tail -3'
))
print(len(os.listdir(WHEELS)), 'wheel files')



72 wheel files


## Serving runtime (decide before running 02)

AgentDojo's local-model path expects an OpenAI-compatible server
(`LOCAL_LLM_PORT`, default 8000). Two options:

- **vLLM**: fast, native tool-call parsing (`vllm_parsed` in AgentDojo).
  Risk: its torch pin can clash with Kaggle's preinstalled torch.
- **llama.cpp server** with GGUF files: smaller dependency footprint,
  runs quantised models on T4.

Pick ONE for all models so the runtime is not a confound. This notebook
does not install a server; cell 4 below only records the choice.


In [4]:
# 4. runtime choice (edit)
RUNTIME = 'vllm'   # 'vllm' or 'llamacpp'
assert RUNTIME in ('vllm', 'llamacpp'), 'choose a runtime first'

In [6]:
# 5. download weights, with a size check BEFORE downloading
from huggingface_hub import snapshot_download, HfApi

token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    pass   # public models still work

api = HfApi()

def bundle_gb():
    """Return the current bundle size in gigabytes."""
    total = 0
    for root, _, names in os.walk(OUT):
        for name in names:
            total += os.path.getsize(os.path.join(root, name))
    return total / 1e9

def repo_gb(model_id, revision=None):
    """Return the total size of a repo's files in gigabytes."""
    info = api.model_info(model_id, revision=revision, token=token,
                          files_metadata=True)
    return sum((f.size or 0) for f in info.siblings) / 1e9

RESOLVED = {}
assert ACTIVE, 'set ACTIVE to the family or families to download now'
for family in ACTIVE:
    spec = CANDIDATES[family]
    assert spec['model_id'], f'fill model_id for {family}'
    need = repo_gb(spec['model_id'], spec['revision'])
    print(family, spec['model_id'], 'repo size GB:', round(need, 1))
    assert bundle_gb() + need < MAX_BUNDLE_GB, (
        f'{need:.1f} GB does not fit. Use a 4-bit build or a smaller model.'
    )
    info = api.model_info(spec['model_id'], revision=spec['revision'],
                          token=token)
    path = snapshot_download(
        spec['model_id'],
        revision=info.sha,
        local_dir=os.path.join(OUT, 'models', family),
        token=token,
    )
    RESOLVED[family] = {
        'model_id': spec['model_id'],
        'revision': info.sha,
        'license': getattr(info, 'card_data', None)
                   and info.card_data.get('license'),
        'path': path,
    }
    print('done, bundle GB:', round(bundle_gb(), 2))

qwen Qwen/Qwen3-14B-AWQ repo size GB: 10.0


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

done, bundle GB: 10.04


In [7]:
# 7. can vLLM be installed here at all? (checks availability only)
import subprocess, sys
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'download', 'vllm', '--no-deps',
     '-d', '/tmp/vllm_test', '-q'],
    capture_output=True, text=True,
)
print('return code:', result.returncode)
print(result.stdout[-500:])
print(result.stderr[-1500:])
print(sh('ls -lh /tmp/vllm_test 2>/dev/null'))

return code: 0


total 228M
-rw-r--r-- 1 root root 228M Nov 23 15:47 vllm-0.34.0-cp38-abi3-manylinux_2_28_x86_64.whl


In [8]:
# 8. total size of vLLM plus all dependencies (measure only)
import os, subprocess, sys
tmp_root = '/kaggle/temp' if os.path.isdir('/kaggle/temp') else '/tmp'
wheel_dir = os.path.join(tmp_root, 'vllm_wheels')
os.makedirs(wheel_dir, exist_ok=True)
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'download', 'vllm',
     '-d', wheel_dir, '-q'],
    capture_output=True, text=True,
)
print('return code:', result.returncode)
print(result.stderr[-1500:])
print(sh(f'du -sh {wheel_dir}; ls {wheel_dir} | wc -l'))

return code: 0

4.0G	/tmp/vllm_wheels
199


In [9]:
# 8b. copy vLLM wheels into the bundle, then test offline resolution
import shutil, subprocess, sys
src = '/tmp/vllm_wheels'
dst = os.path.join(OUT, 'wheels')
os.makedirs(dst, exist_ok=True)
for name in os.listdir(src):
    shutil.copy2(os.path.join(src, name), os.path.join(dst, name))
print('wheels in bundle:', len(os.listdir(dst)))
print('bundle GB:', round(bundle_gb(), 2))

result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--dry-run', '--no-index',
     '--find-links', dst, f'agentdojo=={AGENTDOJO_VERSION}', 'vllm'],
    capture_output=True, text=True,
)
print('offline dry-run return code:', result.returncode)
print(result.stdout[-800:])
print(result.stderr[-1500:])

wheels in bundle: 227
bundle GB: 14.22
offline dry-run return code: 0
45.2 opentelemetry-exporter-otlp-proto-grpc-1.45.2 opentelemetry-exporter-otlp-proto-http-1.45.2 opentelemetry-proto-1.45.2 opentelemetry-sdk-1.45.2 opentelemetry-semantic-conventions-0.66b2 opentelemetry-semantic-conventions-ai-0.5.1 oss-harmony-0.0.12 outlines_core-0.2.14 partial-json-parser-0.2.1.1.post7 prometheus-fastapi-instrumentator-8.1.0 pybase64-1.5.2 pycountry-26.2.16 pydantic-extra-types-2.11.1 pynvvideocodec-2.0.4 quack-kernels-0.6.5 rich-toolkit-0.20.6 rignore-0.8.1 setproctitle-1.3.8 setuptools-80.10.2 sse-starlette-3.7.1 supervisor-4.3.0 tilelang-0.1.12 tokenspeed-mla-2.0.3 tokenspeed-triton-3.9.10.post20261028 torch-2.14.0 torch_c_dlpack_ext-0.1.5 torchcodec-0.17.0 torchvision-0.29.0 triton-3.8.0 types-requests-2.33.0.20261030 vllm-0.34.0 xgrammar-0.2.8 z3-solver-4.15.4.0




In [11]:
# 6. manifest with hashes of every file
import hashlib

assert 'RESOLVED' in globals() and RESOLVED, (
    'run the download cell first and wait for it to finish'
)

def sha256_file(path, block=1 << 20):
    """Return the SHA-256 hex digest of a file."""
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        while chunk := handle.read(block):
            digest.update(chunk)
    return digest.hexdigest()

files = {}
for root, _, names in os.walk(OUT):
    for name in names:
        full = os.path.join(root, name)
        files[os.path.relpath(full, OUT)] = sha256_file(full)

MANIFEST = {
    'env': ENV,
    'agentdojo_version': AGENTDOJO_VERSION,
    'runtime': RUNTIME,
    'models': RESOLVED,
    'pip_freeze': sh('pip freeze').splitlines(),
    'file_sha256': files,
}
with open(os.path.join(OUT, 'MANIFEST.json'), 'w') as handle:
    json.dump(MANIFEST, handle, indent=2)
print('files hashed:', len(files))
print('bundle size GB:', round(bundle_gb(), 2))

files hashed: 266
bundle size GB: 14.22


## After this notebook finishes

1. Commit (Save Version, Save & Run All).
2. Open the output tab and choose **New Dataset** from `bundle/`.
   Keep it private. Name it `targetgate-bundle`.
3. Copy MANIFEST.json into the repository. Its hash goes in the
   registration.
4. Output limit is 20 GB. If the weights are bigger, split into one
   dataset per model family.
